# Shanghai-HOD Benchmark · Colab 一键复现 Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/SH-HOD/blob/main/notebooks/Shanghai_HOD_Colab_Pipeline.ipynb)

> 点击上方徽章即可在 Colab 打开本文件；也可用 Colab「文件 → 打开 notebook → GitHub」按仓库/分支打开。

本 notebook 在 Google Colab 上**端到端复现** Shanghai-HOD 两个评测集，并演示
**基于 MiniMax API 自动完成数据集生成**的完整流程（受事实护栏约束的安全改写）。

| 数据集 | 规模（standard） | 作用 |
|---|---|---|
| **Shanghai-HOD-Q37** | 600 题 | 仅问题的自然语言压力测试：模块路由 / 意图 / 槽位 / 澄清 / 安全拒答 / 抗幻觉 / 口语噪声 |
| **Shanghai-HOD-DataQA37** | 1000 题 | 数据接地 QA：Python 确定性计算答案、证据行可追溯、异常标注、优先级排序、接地播报 |

**架构要点（务必理解）**
1. **确定性内核**：所有隐藏标签、数值、排序、异常、证据行均由 Python 基于 `records.csv` 计算，**不依赖 LLM**。
2. **MiniMax 仅做安全改写**：`--use-litellm` 时，MiniMax 只改写「问题措辞」与「播报语言」；`fact_guard` 强制医院编码 / 数值 / 时间窗 / 指标 / 模块号保持不变，否则回退模板。
3. 因此 **开/关 LLM，隐藏标签与答案完全一致**，仅自然语言表层不同——这正是可复现性的关键。

> **流程**：克隆 → 安装依赖 → 配置 MiniMax 凭证 → API 自检 → 运行生成 → 严格校验 + 单测 → 分布可视化 → 证据可追溯核对 → 评测打分 harness → 打包导出。


## 1️⃣ 克隆仓库并切换分支

In [ ]:
#@title 克隆 / 复用仓库 { display-mode: "form" }
REPO_URL = "https://github.com/pariskang/SH-HOD.git" #@param {type:"string"}
BRANCH   = "main" #@param {type:"string"}
#@markdown > 默认 `main`（已含本 notebook 与强化版生成器）。如需复现某条分支可自行修改；
#@markdown > 若该分支不存在，脚本会自动回退到克隆得到的默认分支。

import os, sys, subprocess
from pathlib import Path

def sh(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    target = Path("/content/SH-HOD")
    if not target.exists():
        sh(["git", "clone", REPO_URL, str(target)])
    sh(["git", "-C", str(target), "fetch", "--all", "--tags", "--quiet"])
    if subprocess.run(["git", "-C", str(target), "checkout", BRANCH]).returncode != 0:
        print(f"⚠️ 分支 {BRANCH!r} 不可用，改用克隆得到的默认分支。")
    REPO_DIR = target
else:
    here = Path.cwd()
    REPO_DIR = next((p for p in [here, *here.parents]
                     if (p / "shanghai_hod_benchmark").exists()), here)

os.chdir(REPO_DIR)
assert (REPO_DIR / "shanghai_hod_benchmark").exists(), f"repo not found at {REPO_DIR}"
_br = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
_sha = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"✅ REPO_DIR = {REPO_DIR}")
print(f"   branch = {_br} | commit = {_sha}")
# Sanity: this notebook expects the strengthened generator/validator.
_vp = REPO_DIR / "shanghai_hod_benchmark/scripts/validate_artifacts.py"
if "cross_window_extremum" not in _vp.read_text(encoding="utf-8"):
    print("⚠️ 该分支缺少强化版生成器/校验器（可能克隆了旧 main）。"
          "请把 BRANCH 设为含本 notebook 的分支后重跑本单元。")

## 2️⃣ 安装依赖

In [ ]:
#@title 安装 litellm / pandas / pyarrow / pytest / matplotlib
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", ".[llm,parquet,dev]", "matplotlib"], check=True)
print("✅ 依赖安装完成")

## 3️⃣ 配置 MiniMax API 凭证

仓库通过 **OpenAI 兼容接口**调用 MiniMax（见 `scripts/litellm_minimax.py`）。三个环境变量：

- `MINIMAX_API_KEY` —— **必填**（启用 LLM 改写时）；
- `MINIMAX_API_BASE` —— OpenAI 兼容 base_url（MiniMax 官方或你的自建中转）；
- `MINIMAX_MODEL` —— 默认 `MiniMax-M1`。

> 🔐 **安全**：推荐用 Colab 左侧 **🔑 Secrets** 添加 `MINIMAX_API_KEY` 并对本 notebook 授权；
> Key 只写入运行时环境变量，**不会**写入仓库或被提交。

In [ ]:
#@title 写入凭证（Secrets / 弹窗输入） { display-mode: "form" }
MINIMAX_MODEL    = "MiniMax-M1" #@param {type:"string"}
MINIMAX_API_BASE = "https://api.minimaxi.com/v1" #@param {type:"string"}
#@markdown > 若用自建 OpenAI 兼容中转，请把 `MINIMAX_API_BASE` 改成你的 base_url。

import os
os.environ["MINIMAX_MODEL"] = MINIMAX_MODEL
os.environ["MINIMAX_API_BASE"] = MINIMAX_API_BASE

key = os.environ.get("MINIMAX_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("MINIMAX_API_KEY")
    except Exception:
        key = None
if not key:
    import getpass
    key = getpass.getpass("输入 MINIMAX_API_KEY（留空=跳过 LLM，使用确定性复现）：").strip()

if key:
    os.environ["MINIMAX_API_KEY"] = key
    print(f"✅ MINIMAX_API_KEY 已配置（{key[:4]}…，长度 {len(key)}）")
    print(f"   base = {MINIMAX_API_BASE}  |  model = {MINIMAX_MODEL}")
else:
    print("⚠️ 未提供 API Key —— 可继续用确定性（无 LLM）复现，隐藏标签/数值完全一致。")

## 4️⃣ MiniMax API 连通性自检

生成脚本里 `rewrite_question` / `completion_json` 在 Key 缺失或失效时会**抛异常并中断生成**。
因此这里先做一次**真实调用**：成功才把 `API_OK=True`，后续生成才会带 `--use-litellm`；
否则自动回退到确定性生成，保证 notebook 不会中途崩溃。

In [ ]:
#@title API 自检 + 事实护栏演示
import os, sys
sys.path.insert(0, str(REPO_DIR / "shanghai_hod_benchmark" / "scripts"))
from litellm_minimax import (LiteLLMConfig, completion_json,
                             litellm_available, rewrite_question, fact_guard)

API_OK = False
print("litellm installed:", litellm_available())

if os.environ.get("MINIMAX_API_KEY") and litellm_available():
    # 显式传参：LiteLLMConfig 的默认值在 import 时即固化，必须显式覆盖
    cfg = LiteLLMConfig(model=os.environ.get("MINIMAX_MODEL", "MiniMax-M1"),
                        api_base=os.environ.get("MINIMAX_API_BASE"),
                        api_key=os.environ.get("MINIMAX_API_KEY"))
    print("→ litellm model:", cfg.litellm_model, "| base:", cfg.api_base)
    try:
        r = completion_json(
            [{"role": "system", "content": "You only output a JSON object."},
             {"role": "user", "content": '返回 {"status": "ok"}'}], cfg)
        print("✅ API 调用成功：", r)
        API_OK = True
    except Exception as e:
        print("❌ API 调用失败：", repr(e))
        print("   → 回退确定性生成（隐藏标签/数值不受影响，仅问题措辞不经 LLM 改写）。")
else:
    print("ℹ️ 无 Key 或未安装 litellm → 使用确定性生成。")

# 事实护栏：改写只允许动措辞，事实（医院号/数值/时间/指标/模块）必须一致
if API_OK:
    demo = "2026-06-03 16:00，SH-MH002 在 M02 模块的门急诊量是多少？"
    try:
        rw = rewrite_question(demo, {})
        print("\n原始：", demo)
        print("改写：", rw)
        print("fact_guard 事实一致：", fact_guard(demo, rw))
    except Exception as e:
        print("rewrite 演示失败：", repr(e))

## 5️⃣ 运行生成 Pipeline

`generate_benchmarks.py` 会重建全部 artifacts。勾选 `USE_LITELLM` 且自检通过时，
**MiniMax** 负责改写问题措辞 / 润色播报语（受 `fact_guard` 约束）。

> ⏱️ **时间 / 成本**：`standard` + LLM 首次约发起 ~1600 次**带磁盘缓存**的调用（`.cache/shanghai_hod_litellm`），
> 较慢且产生费用；**重跑因缓存命中而很快**。确定性模式秒级完成，且隐藏标签/数值与提交版完全一致。
>
> 计数填 `0` = 用 profile 默认值。**严格 `validate` / `pytest` 需要 `standard` 默认计数。**

In [ ]:
#@title 生成数据集 { display-mode: "form" }
PROFILE      = "standard" #@param ["mini", "standard", "full"]
USE_LITELLM  = True #@param {type:"boolean"}
Q37_COUNT    = 0 #@param {type:"integer"}
DATAQA_COUNT = 0 #@param {type:"integer"}

import sys, subprocess
use_llm = USE_LITELLM and API_OK
if USE_LITELLM and not API_OK:
    print("⚠️ 勾选了 USE_LITELLM 但 API 自检未通过 → 本次使用确定性生成。\n")

cmd = [sys.executable,
       str(REPO_DIR / "shanghai_hod_benchmark/scripts/generate_benchmarks.py"),
       "--profile", PROFILE]
if use_llm:
    cmd.append("--use-litellm")
if Q37_COUNT > 0:
    cmd += ["--q37-count", str(Q37_COUNT)]
if DATAQA_COUNT > 0:
    cmd += ["--dataqa-questions", str(DATAQA_COUNT)]

print("$", " ".join(cmd))
print("（MiniMax 改写中，首次较慢…）" if use_llm
      else "（确定性模式：秒级完成，结果与提交版一致。）")
subprocess.run(cmd, check=True, env=os.environ.copy())
print("\n✅ 生成完成")

## 6️⃣ 严格校验 + 单元测试

`validate_artifacts.py` 做跨文件强校验：题型覆盖（DataQA 15 类 / Q37 11 类）、
难度分布（hard ≥ 40%、easy ≤ 25%）、幻觉陷阱多样性（≥ 12 种）、播报事实接地（0 处越界引用）、
padding 占比（≤ 40%）等。

> ⚠️ 这些阈值是按 **standard** 设计的；若上一步选了 `mini`，部分阈值可能不满足属正常现象。

In [ ]:
#@title 运行 validate_artifacts.validate() + pytest
import importlib, sys, json
sys.path.insert(0, str(REPO_DIR / "shanghai_hod_benchmark/scripts"))
import validate_artifacts; importlib.reload(validate_artifacts)

report = validate_artifacts.validate(REPO_DIR / "shanghai_hod_benchmark")
print("✅ 校验通过\n")
print(json.dumps(report, ensure_ascii=False, indent=2))

import subprocess
print("\n— pytest —")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], cwd=str(REPO_DIR))

## 7️⃣ 分布概览与可视化

In [ ]:
#@title 难度 / 题型 / 任务类型分布
import json, pandas as pd, matplotlib.pyplot as plt
B = REPO_DIR / "shanghai_hod_benchmark"

def load(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

hidden = load(B / "dataset_1_question_only/questions_with_hidden_metadata.jsonl")
dqa    = load(B / "dataset_2_data_qa/questions.jsonl")
hq, dq = pd.DataFrame(hidden), pd.DataFrame(dqa)
print(f"Q37 = {len(hq)}   DataQA = {len(dq)}")

traps = [h["expected_slots"].get("trap_type")
         for h in hidden if h["question_type"] == "hallucination_trap"]
print("hallucination trap 去重种类:", len({t for t in traps if t}))

display(hq["difficulty"].value_counts().rename("count").to_frame())
display(hq["question_type"].value_counts().rename("count").to_frame())
display(dq["task_type"].value_counts().rename("count").to_frame())

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
hq["difficulty"].value_counts().reindex(["easy", "medium", "hard"]).plot.bar(
    ax=ax[0], title="Q37 difficulty", color="#4C72B0")
hq["question_type"].value_counts().plot.bar(
    ax=ax[1], title="Q37 question_type", color="#55A868")
dq["task_type"].value_counts().plot.bar(
    ax=ax[2], title="DataQA task_type", color="#C44E52")
for a in ax:
    a.tick_params(axis="x", rotation=80)
plt.tight_layout(); plt.show()

## 8️⃣ 样例与可追溯证据（`evidence_rows` ↔ `records.csv`）

In [ ]:
#@title 抽样展示一道困难 Q37 与一道可核对的 DataQA
import csv, json
B = REPO_DIR / "shanghai_hod_benchmark"
answers = {a["question_id"]: a for a in load(B / "dataset_2_data_qa/answers.jsonl")}
recs = {r["row_id"]: r for r in csv.DictReader(
    open(B / "dataset_2_data_qa/records.csv", encoding="utf-8"))}

chain = [h for h in hidden
         if h["difficulty"] == "hard" and h["question_type"] == "cross_module_chain"]
ex = chain[0] if chain else [h for h in hidden if h["difficulty"] == "hard"][0]
print("【困难 Q37 ·", ex["question_type"], "】")
print(" Q:", ex["question"])
print(" target_module:", ex["target_module"], "| query_type:", ex["expected_query_type"])
print(" expected_slots:", json.dumps(ex.get("expected_slots", {}), ensure_ascii=False))

cand = [q for q in dqa if q["task_type"] == "cross_window_extremum"]
q = cand[0]; a = answers[q["question_id"]]
print("\n【DataQA · cross_window_extremum】")
print(" Q:", q["question"])
print(" answer_value:", json.dumps(a["answer_value"], ensure_ascii=False))
print(" calculation :", a["calculation"], "| confidence:", a["confidence"])
print(" evidence_rows:", a["evidence_rows"])
print(" —— 逐行核对证据（直接来自 records.csv）——")
for rid in a["evidence_rows"][:6]:
    r = recs[rid]
    print(f"   {rid}: {r['hospital_id']} {r['indicator_code']} "
          f"{r['timestamp_start']} = {r['value']}{r['unit']} flag={r['data_quality_flag']}")

## 9️⃣ 评测打分 harness 演示（`scorer.py`）

用 gold 当「完美预测」可看到指标接近满分；再用「打乱模块」的退化预测，验证指标确实有区分度。
真实评测时，把模型预测写成相同 JSONL 格式喂给 `dataset1_score` / `dataset2_score` 即可。

In [ ]:
#@title 完美预测 vs 退化预测
import importlib, sys, json, tempfile, random
from pathlib import Path
sys.path.insert(0, str(REPO_DIR / "shanghai_hod_benchmark/evaluation"))
import scorer; importlib.reload(scorer)

B = REPO_DIR / "shanghai_hod_benchmark"
tmp = Path(tempfile.mkdtemp())

def dump(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

# Q37：gold 当完美预测
goldQ = B / "dataset_1_question_only/questions_with_hidden_metadata.jsonl"
perfectQ = dump([{ "question_id": h["question_id"], "target_module": h["target_module"],
    "expected_query_type": h["expected_query_type"], "expected_slots": h.get("expected_slots", {}),
    "requires_clarification": h.get("requires_clarification", False),
    "should_refuse": h.get("should_refuse", False)} for h in hidden], tmp / "q37_perfect.jsonl")
print("Q37 完美预测：")
print(json.dumps(scorer.dataset1_score(goldQ, perfectQ), ensure_ascii=False, indent=2))

# Q37：打乱 module 的退化预测
mods = [h["target_module"] for h in hidden]; random.seed(0); random.shuffle(mods)
badQ = dump([{ "question_id": h["question_id"], "target_module": m,
    "expected_query_type": "DATA_QUERY", "expected_slots": {}}
    for h, m in zip(hidden, mods)], tmp / "q37_bad.jsonl")
print("\nQ37 退化预测（打乱模块）：")
print(json.dumps(scorer.dataset1_score(goldQ, badQ), ensure_ascii=False, indent=2))

# DataQA：gold 当完美预测
goldA = B / "dataset_2_data_qa/answers.jsonl"
ans = load(goldA)
perfectA = dump([{ "question_id": a["question_id"], "answer_value": a.get("answer_value"),
    "evidence_rows": a.get("evidence_rows", []), "final_answer": a.get("final_answer", "")}
    for a in ans], tmp / "dqa_perfect.jsonl")
print("\nDataQA 完美预测：")
print(json.dumps(scorer.dataset2_score(goldA, perfectA), ensure_ascii=False, indent=2))

## 🔟 打包下载 / 导出（可选）

In [ ]:
#@title 打包 artifacts；Colab 自动触发下载
import shutil, sys
base = "/content/shanghai_hod_artifacts" if "google.colab" in sys.modules \
       else str(REPO_DIR / "shanghai_hod_artifacts")
zip_path = shutil.make_archive(base, "zip", str(REPO_DIR / "shanghai_hod_benchmark"))
print("✅ 已打包:", zip_path)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path)

#@markdown 可选：本地导出 Parquet（仓库不提交二进制）：
#@markdown ```
#@markdown !python shanghai_hod_benchmark/scripts/generate_benchmarks.py \
#@markdown     --profile standard --export-parquet /content/records.parquet
#@markdown ```

## 📎 附录

### Profiles
| profile | hospitals | days | slots | Q37 | DataQA | 用途 |
|---|---|---|---|---|---|---|
| `mini` | 3 | 1 | 4 | 300 | 120 | 快速冒烟，**不满足严格 validate 阈值** |
| `standard` | 37 | 1 | 8 | 600 | 1000 | **仓库提交参考集**，validate / pytest 基准 |
| `full` | 37 | 7 | 48 | 1000 | 3000 | 生产级大规模本地物化 |

### 成本 / 性能
- MiniMax 调用按内容哈希落盘缓存（`.cache/shanghai_hod_litellm`），重跑命中缓存近乎免费。
- 想低成本体验 LLM 改写：保持 `standard` 结构，把 `Q37_COUNT` / `DATAQA_COUNT` 调小（如 30）。
  注意小计数下严格 `validate` 可能因题型覆盖不全而失败，属预期。

### 常见问题
- **API 自检失败**：确认 `MINIMAX_API_BASE` 为 OpenAI 兼容 `/v1`、Key 有效、模型名正确；
  自检失败会自动回退确定性生成，artifacts 仍有效。
- **validate 断言失败**：多因用了 `mini` 或自定义小计数；改回 `standard` 默认计数即可。
- **JSON mode 不支持**：自检用 `response_format=json_object`，若中转不支持会判定失败并回退。

### 安全与隐私
- 数值 / 标签 / 排序 / 证据**永不**由 LLM 生成，全部 Python 确定性计算。
- `data_sources.ingest_csv` 拒绝患者级字段并默认匿名化医院 ID。
- API Key 仅存于运行时环境变量，不写入仓库。
